# ML-07 — Baseline Action Score and Top-20 Review

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassanraza04/flyrank_intern_content/blob/main/work/notebooks/w04_baseline_score.ipynb)

This notebook builds the transparent baseline for my Refresh / Content Opportunity Scoring lane. It uses only GSC signals known before the later outcome window.

## Before running

In Colab, add your Hugging Face Read token as the `HF_TOKEN` secret and enable notebook access. The token is never saved in this notebook or committed to GitHub. The first full scan can take several minutes, so this notebook caches only the page-level feature frame locally for the current runtime.

In [1]:
%pip -q install duckdb pandas pyarrow scikit-learn matplotlib

from pathlib import Path

if not Path('work/scripts/capstone_data.py').exists():
    !git clone -q --branch capstone-refresh https://github.com/hassanraza04/flyrank_intern_content.git
    %cd flyrank_intern_content


/content/flyrank_intern_content


In [2]:
import os
from pathlib import Path

import pandas as pd

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except ImportError:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    raise RuntimeError('Set HF_TOKEN in Colab Secrets or the local environment. Do not paste it into this notebook.')

from work.scripts.capstone_data import build_feature_frame, create_connection
from work.scripts.capstone_utils import add_baseline_score, evaluate_ranking, reason_codes

CACHE_PATH = Path('work/outputs/capstone_features.parquet')
connection = create_connection(HF_TOKEN)
print('DuckDB connection ready. The feature query uses the full fact table, never the June sample table.')

DuckDB connection ready. The feature query uses the full fact table, never the June sample table.


## 1. My rule and its reason codes

My baseline prioritizes a page when its search impressions have already fallen relative to the preceding 28 days and the page still has meaningful current search visibility. A worsening average position adds a small amount to the score. The reason codes are recent search momentum down, average position worsened, meaningful search visibility, low CTR review candidate, or monitor for more evidence. This is a transparent review queue, not an automatic edit instruction.

## 2. Build the ranked queue

The two prior 28-day windows form the feature history. The later 28-day window creates the proxy label. The June outcome cohort is not opened or used to tune this rule.

In [3]:
if CACHE_PATH.exists():
    features = pd.read_parquet(CACHE_PATH)
    print('Loaded the cached aggregate feature frame.')
else:
    features = build_feature_frame(connection, CACHE_PATH)
    print('Built and cached the aggregate feature frame.')

cohort_summary = features.groupby('cohort_id')['is_declining_proxy'].agg(rows='size', declining_proxy_rate='mean').reset_index()
cohort_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Built and cached the aggregate feature frame.


,cohort_id,rows,declining_proxy_rate
0,2025-12,23302,0.159385
1,2026-01,27630,0.267354
2,2026-02,31692,0.219708
3,2026-03,35992,0.138864
4,2026-04,44063,0.222636
5,2026-05,50735,0.446575
6,2026-06-sealed,46898,0.518935


In [4]:
development = features.query("cohort_id != '2026-06-sealed'").copy()
validation = add_baseline_score(development.query("cohort_id == '2026-05'").copy())
validation['reason_codes'] = validation.apply(reason_codes, axis=1)

baseline_metrics = evaluate_ranking(
    validation['is_declining_proxy'], validation['baseline_score'], k=100
)
print({'baseline_development_metrics': baseline_metrics})

local_queue = validation.sort_values('baseline_score', ascending=False).copy()
local_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print('Wrote a local-only baseline queue. It is intentionally ignored by git.')

{'baseline_development_metrics': {'base_rate': 0.4465753424657534, 'precision_at_k': 0.69, 'roc_auc': 0.5216837141830102}}
Wrote a local-only baseline queue. It is intentionally ignored by git.


## 3. Top-20 review

The top 20 are for human review. To keep this public notebook safe, I summarize their actions and reason codes rather than display client or content identifiers. A reviewer should inspect the underlying page context privately before making any change.

In [5]:
top_20 = local_queue.head(20).copy()
top_20_reason_counts = top_20.explode('reason_codes')['reason_codes'].value_counts().rename_axis('reason_code').reset_index(name='top_20_count')
top_20_reason_counts

,reason_code,top_20_count
0,recent_search_momentum_down,20
1,low_ctr_review_candidate,20
2,average_position_worsened,19
3,meaningful_search_visibility,7


## 4. Weak picks and leakage check

The rule can be wrong when demand or seasonality changes for reasons unrelated to the page, or when an observed position change is noisy. It does not use the later-window impressions, the proxy label, any existing product score, query text, GA4 engagement signals, or IDs. Those exclusions keep the rule available at the decision moment.

In [6]:
forbidden_columns = {'next_impressions', 'is_declining_proxy', 'client_hash_id', 'content_hash_id'}
baseline_inputs = {'current_impressions', 'impression_change_pct', 'position_change'}
assert forbidden_columns.isdisjoint(baseline_inputs)
print('Leakage guard passed:', sorted(baseline_inputs))

Leakage guard passed: ['current_impressions', 'impression_change_pct', 'position_change']


## Self-check

- [x] The rule is readable and has reason codes.
- [x] The baseline uses the same proxy label and development cohort that the model will use.
- [x] The final June outcome cohort remains sealed.
- [x] The notebook displays no client names, domains, URLs, raw queries, or identifiers.
- [ ] I have run this notebook in Colab and saved the executed version to GitHub.
